# Train Experiment

This notebook calls the same `training.run_experiment` code used by the CLI.

The SI split holds out complete participants for fixed validation. Training,
validation, and test features stream from CPU/disk in small batches while the
unchanged GCN computation remains on GPU. This run uses `hand_body` as the
fair SI GCN threshold baseline.

In [ ]:
import gc
import os
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "torch")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import keras
import torch

from training.config import ExperimentConfig
from training.data import load_split_records
from training.experiment import run_experiment
from training.scenario_splits import build_training_validation_split

keras.backend.clear_session()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU free before training: {free_bytes / 1024**3:.2f}/{total_bytes / 1024**3:.2f} GiB")

CUDA available: True
GPU free before training: 14.80/15.93 GiB


In [ ]:
DEFAULT_CONFIG = ExperimentConfig()

# Fair full-dataset SI GCN hand+body baseline.
# Data remains on CPU and is transferred one small batch at a time.
SCENARIO = "si"
MODEL = "gcn"
MODALITY = "body_hand_face"
FRAME_SAMPLE_COUNT = DEFAULT_CONFIG.frame_sample_count
EPOCHS = DEFAULT_CONFIG.epochs
BATCH_SIZE = DEFAULT_CONFIG.batch_size
EVALUATION_BATCH_SIZE = DEFAULT_CONFIG.evaluation_batch_size
NUM_WORKERS = 0
CACHE_DEVICE = "cpu"
TRAIN_PARTITIONS = DEFAULT_CONFIG.train_partitions
PARTITION_CYCLES = DEFAULT_CONFIG.partition_cycles
TEST_PARTITIONS = DEFAULT_CONFIG.test_partitions
VALIDATE_DURING_TRAINING = True
VALIDATION_FRACTION = DEFAULT_CONFIG.validation_fraction
LEARNING_RATE = DEFAULT_CONFIG.learning_rate
PARTITION_LEARNING_RATE = DEFAULT_CONFIG.partition_learning_rate
OPTIMIZER = DEFAULT_CONFIG.optimizer
EARLY_STOPPING_PATIENCE = DEFAULT_CONFIG.early_stopping_patience
EARLY_STOPPING_MIN_CYCLES = DEFAULT_CONFIG.early_stopping_min_cycles
EARLY_STOPPING_MIN_DELTA = DEFAULT_CONFIG.early_stopping_min_delta
SEED = DEFAULT_CONFIG.seed
DATASET_DIR = DEFAULT_CONFIG.dataset_dir
ARTIFACTS_DIR = DEFAULT_CONFIG.artifacts_dir

print(
    f"scenario={SCENARIO} model={MODEL} modality={MODALITY} "
    f"batch={BATCH_SIZE} eval_batch={EVALUATION_BATCH_SIZE} "
    f"cache={CACHE_DEVICE} partitions={TRAIN_PARTITIONS} cycles={PARTITION_CYCLES}"
)

official_train_records = load_split_records(DATASET_DIR, SCENARIO, "train")
split_preview = build_training_validation_split(
    official_train_records,
    scenario=SCENARIO,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
)
split_preview.to_dict()

scenario=si model=gcn modality=body_hand_face batch=8 eval_batch=8 cache=cpu partitions=6 cycles=50


{'strategy': 'si_participant_disjoint',
 'training_size': 40366,
 'validation_size': 10092,
 'validation_fraction': 0.200007927385152,
 'training_class_count': 2731,
 'validation_class_count': 2731,
 'training_participants': ['P1',
  'P11',
  'P12',
  'P13',
  'P14',
  'P16',
  'P19',
  'P2',
  'P20',
  'P21',
  'P24',
  'P25',
  'P26',
  'P27',
  'P28',
  'P29',
  'P3',
  'P31',
  'P32',
  'P33',
  'P34',
  'P36',
  'P37',
  'P39',
  'P4',
  'P41',
  'P43',
  'P44',
  'P45',
  'P46',
  'P5',
  'P50',
  'P7'],
 'validation_participants': ['P10',
  'P23',
  'P30',
  'P38',
  'P40',
  'P51',
  'P52',
  'P8'],
 'participant_overlap': [],
 'classes_missing_from_training': [],
 'classes_missing_from_validation': []}

In [ ]:
config = ExperimentConfig(
    scenario=SCENARIO,
    model=MODEL,
    modality=MODALITY,
    frame_sample_count=FRAME_SAMPLE_COUNT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    cache_device=CACHE_DEVICE,
    train_partitions=TRAIN_PARTITIONS,
    partition_cycles=PARTITION_CYCLES,
    validate_during_training=VALIDATE_DURING_TRAINING,
    validation_fraction=VALIDATION_FRACTION,
    evaluation_batch_size=EVALUATION_BATCH_SIZE,
    test_partitions=TEST_PARTITIONS,
    optimizer=OPTIMIZER,
    learning_rate=LEARNING_RATE,
    partition_learning_rate=PARTITION_LEARNING_RATE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_cycles=EARLY_STOPPING_MIN_CYCLES,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    seed=SEED, 
    dataset_dir=DATASET_DIR,
    artifacts_dir=ARTIFACTS_DIR,
)

result = run_experiment(config)
result["summary"]

[runtime] validation_strategy=si_participant_disjoint training_samples=40366 validation_samples=10092; test data streams from CPU in 3 partitions with batch_size=8 num_workers=0
[runtime] keras_backend=torch
[runtime] requested_device=gpu
[runtime] batch_size=8 num_workers=0
[runtime] cache_device=cpu
[runtime] torch_cuda_available=True
[runtime] torch_cuda_device_count=1
[runtime] torch_cuda_current_device=0
[runtime] torch_cuda_device_name=NVIDIA GeForce RTX 5060 Ti
[runtime] torch_cuda_memory_allocated_mb=0.00
[runtime] torch_cuda_memory_reserved_mb=0.00
[runtime] keras_device_scope=gpu:0
[runtime] optimizer_learning_rate=0.0001
[runtime] partition_cycle=1/50
[runtime] training cycle=1/50 partition=1/6 train_samples=6728 epochs=2
[runtime][train_begin] cuda_memory_allocated_mb=11.37
[runtime][train_begin] cuda_memory_reserved_mb=22.00
[runtime][epoch_1_begin] cuda_memory_allocated_mb=11.37
[runtime][epoch_1_begin] cuda_memory_reserved_mb=22.00
Epoch 1/2
840/841 ━━━━━━━━━━━━━━━━━━━━ 

{'scenario': 'si',
 'model': 'gcn',
 'modality': 'body_hand_face',
 'run_dir': 'artifacts/experiments/si/gcn/body_hand_face/20260614T020157Z',
 'input_shape': [30, 2765],
 'num_classes': 2731,
 'train_size': 40366,
 'validation_size': 10092,
 'test_size': 32941,
 'validation_split': {'strategy': 'si_participant_disjoint',
  'training_size': 40366,
  'validation_size': 10092,
  'validation_fraction': 0.200007927385152,
  'training_class_count': 2731,
  'validation_class_count': 2731,
  'training_participants': ['P1',
   'P11',
   'P12',
   'P13',
   'P14',
   'P16',
   'P19',
   'P2',
   'P20',
   'P21',
   'P24',
   'P25',
   'P26',
   'P27',
   'P28',
   'P29',
   'P3',
   'P31',
   'P32',
   'P33',
   'P34',
   'P36',
   'P37',
   'P39',
   'P4',
   'P41',
   'P43',
   'P44',
   'P45',
   'P46',
   'P5',
   'P50',
   'P7'],
  'validation_participants': ['P10',
   'P23',
   'P30',
   'P38',
   'P40',
   'P51',
   'P52',
   'P8'],
  'participant_overlap': [],
  'classes_missing_from_tr